# BorakBot — Try Whisper (10 minutes, no local install)

Run this on **Google Colab** or **Kaggle Notebooks**. Runtime > Change runtime type > **T4 GPU**
(CPU works too, just slower).

Goal: hear Whisper transcribe your own Bahasa Rojak, and compare two models:

| Model | What it is |
|---|---|
| `openai/whisper-small` | Vanilla Whisper — what proposal §5.6 specifies |
| `mesolitica/malaysian-whisper-small-v3` | Same size, fine-tuned on Malay + **Manglish** |

No WER, no test set yet. Just listen to the difference.

In [ ]:
!pip install -q openai-whisper transformers
!apt-get -qq install -y ffmpeg > /dev/null
print("ready")

## 1. Get some audio

**Option A — smoke test.** A Malay speech clip, to confirm the pipeline works at all
before you blame the model for anything.

In [ ]:
!wget -q -O sample.mp3 https://github.com/mesolitica/malaya-speech/raw/master/speech/assembly.mp3
AUDIO = "sample.mp3"

from IPython.display import Audio
Audio(AUDIO)

**Option B — your own rojak.** Record 5-10 seconds on your phone, then upload.

Say something with a real mid-sentence switch, e.g.
*"eh macam mana nak check balance akaun ni lah"* — that is where Whisper is predicted
to struggle (§5.6), so it is the interesting case, not the easy one.

Uncomment and run:

In [ ]:
# from google.colab import files          # Colab only
# up = files.upload()
# AUDIO = list(up.keys())[0]
# print("using:", AUDIO)

## 2. Vanilla Whisper small

This is the `openai-whisper` package your proposal cites (Radford et al., 2023).

`language=None` lets Whisper auto-detect. `fp16=False` avoids a warning on CPU.

In [ ]:
import whisper

base = whisper.load_model("small")          # ~461 MB download, cached after first run
out_base = base.transcribe(AUDIO, language=None, fp16=False)

print("detected language:", out_base["language"])
print("transcript:", out_base["text"].strip())

### Try nudging it

`initial_prompt` is fed to the decoder as *preceding context*, not as an instruction —
it biases spelling and register toward whatever you show it. Often the cheapest WER win
available. It will not obey commands, so do not write it like a system prompt.

In [ ]:
ROJAK_PROMPT = (
    "Ini perbualan Bahasa Rojak Malaysia. Contoh: saya nak check balance akaun boleh "
    "tak, macam mana nak renew lesen ni lah, harga dia RM2.50 je."
)

print("no prompt   :", base.transcribe(AUDIO, fp16=False)["text"].strip())
print("with prompt :", base.transcribe(AUDIO, fp16=False, initial_prompt=ROJAK_PROMPT)["text"].strip())
print("forced ms   :", base.transcribe(AUDIO, fp16=False, language="ms")["text"].strip())
print("forced en   :", base.transcribe(AUDIO, fp16=False, language="en")["text"].strip())

## 3. Malaysian-fine-tuned Whisper

Same `small` size class, distilled from Whisper Large v3 on Malaysian data and
explicitly trained on Manglish. Loaded through `transformers`, not the `whisper`
package — different library, same underlying architecture.

If v3 misbehaves, its model card notes stage-2 training was still in progress; fall
back to `mesolitica/malaysian-whisper-small-v2`.

In [ ]:
import torch
from transformers import pipeline

MODEL_ID = "mesolitica/malaysian-whisper-small-v3"
device = 0 if torch.cuda.is_available() else -1

asr = pipeline(
    "automatic-speech-recognition",
    model=MODEL_ID,
    device=device,
    torch_dtype=torch.float32,   # float32 is safe everywhere; model is only 0.2B
    chunk_length_s=30,           # split long audio into 30s windows.
                                 # Without this, transformers routes anything >30s to
                                 # long-form generation, which REQUIRES timestamp tokens
                                 # and raises ValueError. Chunking keeps us on the simple
                                 # short-form path and returns clean text.
)

out_my = asr(AUDIO, generate_kwargs={"language": "ms", "task": "transcribe"})
print("transcript:", out_my["text"].strip())

# Alternative if you WANT long-form (single pass, better cross-chunk context):
#   asr_lf = pipeline("automatic-speech-recognition", model=MODEL_ID, device=device,
#                     torch_dtype=torch.float32)          # no chunk_length_s
#   out_my = asr_lf(AUDIO, return_timestamps=True,
#                   generate_kwargs={"language": "ms", "task": "transcribe"})


## 4. Side by side

In [ ]:
print("VANILLA   :", out_base["text"].strip())
print()
print("MALAYSIAN :", out_my["text"].strip())

## What to look for

Read both transcripts against what you actually said, and note:

1. **Did the English words survive as English?** Vanilla Whisper commits to one language
   per segment, so `check` can come back as `cek`. This is the §5.6 limitation showing up
   in your own data.
2. **Did the particles survive?** *lah / lor / meh / kan* are frequently dropped or
   rewritten — they carry pragmatic meaning your chatbot needs.
3. **Malaysian entities.** JPJ, MyJPJ, Touch 'n Go, EPF, RM amounts.
4. **Latency.** Time both. Demo usability is a real constraint, not just accuracy.

If the Malaysian model is clearly better on rojak, that is your justification for
switching — and the vanilla-vs-fine-tuned comparison becomes your speech evaluation
almost for free.

**Next**: once you have a feel for it, build the labelled test set and measure WER
properly — see `docs/whisper_step_by_step.md`.

---

## 5. Model-card style: raw processor + word-level timestamps

Section 3 used `pipeline()`, which hides the tokeniser and returns clean text. The
model card instead drives `processor` and `model.generate` directly. Use this version
when you want **word-level timestamps** (the card's custom `transcribeprecise` task),
which `pipeline` cannot give you.

Three fixes applied to the card's code so it runs on a free Colab/Kaggle T4:

1. **Model ID** — the card's snippet loads `malaysia-ai/malaysian-whisper-small`, which
   looks like a leftover from their training repo. Use the repo you actually want.
2. **`.cuda()`** — crashes on a CPU runtime. Detect the device instead.
3. **`bfloat16`** — T4 is Turing (compute 7.5) with no native bf16; some kernels error
   or run slowly. `float16` on GPU, `float32` on CPU.

In [ ]:
import torch, requests, io, numpy as np, soundfile as sf

# Must be patched BEFORE WhisperForConditionalGeneration is imported -- it registers the
# custom `transcribeprecise` task ID that this checkpoint was trained with.
from transformers.models.whisper import tokenization_whisper
tokenization_whisper.TASK_IDS = ["translate", "transcribe", "transcribeprecise"]

from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_ID = "mesolitica/malaysian-whisper-small-v3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

processor = WhisperProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE
).to(DEVICE).eval()

print("loaded on", DEVICE, DTYPE)

Load audio. The card uses `datasets.Audio` to decode; `soundfile` is more direct and
one less dependency. Whisper requires **16 kHz mono float32**, hence the resample check.

In [ ]:
import librosa

y, _ = librosa.load(AUDIO, sr=16000, mono=True)   # librosa resamples for you
print("duration:", round(len(y) / 16000, 1), "s")

# NOTE: raw processor + generate handles ONE 30-second window only. Anything longer is
# silently truncated. pipeline(chunk_length_s=30) and the openai-whisper package chunk
# automatically; this path does not.
if len(y) > 16000 * 30:
    print("WARNING: longer than 30s, will be truncated")

In [ ]:
def transcribe_raw(y, task="transcribe", language="ms", clean=True):
    with torch.no_grad():
        p = processor([y], sampling_rate=16000, return_tensors="pt")
        feats = p["input_features"].to(DEVICE, DTYPE)
        out = model.generate(
            feats,
            language=language,
            task=task,              # "transcribe" | "transcribeprecise" | "translate"
            return_timestamps=True,   # required once audio is long-form
            return_dict_in_generate=True,
            # output_scores=True,   # card sets this; only needed for token confidence
        )
    ids = out.sequences[0]
    if clean:
        # skip_special_tokens strips <|startoftranscript|>, <|ms|>, timestamps etc.
        return tokenizer.decode(ids, skip_special_tokens=True).strip()
    # keep special tokens -- this is what the model card prints
    return tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(ids))


print("CLEAN TEXT:\n", transcribe_raw(y), "\n")
print("WITH TOKENS (card output):\n", transcribe_raw(y, clean=False), "\n")
print("WORD TIMESTAMPS:\n", transcribe_raw(y, task="transcribeprecise", clean=False))

**For BorakBot, use `clean=True`.** The normaliser and the LLM need plain text —
feeding them `<|startoftranscript|><|ms|><|0.02|>` would break Stage 2 immediately.

Timestamps are not needed for this project. They are worth knowing about only if you
later want to highlight words in the UI as they are recognised, which is out of scope.